# GNN→MLP distillation on Roman-empire / Amazon-ratings (Colab GPU)

Lightweight GLNN-style logit distillation with graph morphing, extended to two larger heterophily
benchmarks. 5 variants: `MLP_X`, `KD_orig_X`, `KD_morph_X`, `KD_morph_X_HashAdj`, `KD_morph_X_BloomAdj`.

**Design constraints**: teacher input is always **raw X** (never HashAdj/BloomAdj/LCE); edge morphing
comes from **LCC pseudo-labels** (`teacher_morph = GCN(X, E_orig ∪ E_LCC-allpseudo)`); HashAdj/BloomAdj
are **label-free student-only** features. KD = `CE(train) + λ·KL(all nodes; T=2)`, λ∈{1,2}.

Run each dataset cell separately (they are heavy). Set `QUICK=True` for a seed-0 smoke first.
**Runtime → Change runtime type → GPU** before running.

In [1]:
# Cell 1: GPU / CUDA check
!nvidia-smi -L
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda:', torch.version.cuda)
if not torch.cuda.is_available():
    print('WARNING: no GPU — Roman-empire/Amazon-ratings will be very slow. Runtime → change to GPU.')

GPU 0: Tesla T4 (UUID: GPU-a56ae807-b81a-2db1-c708-bba439049592)
torch: 2.11.0+cu128
cuda available: True
cuda: 12.8


In [4]:
# Cell 2: repository setup — pick ONE option
# Option A: clone from GitHub (private OnizukaLab/graphMorphing → use a PAT in the URL)
!git clone https://github.com/OnizukaLab/graphMorphing.git
SRC = '/content/graphMorphing/src'

# Option B: Google Drive (upload the graphMorphing repo there)
from google.colab import drive
drive.mount('/content/drive')
#SRC = '/content/drive/MyDrive/graphMorphing/src'  # <-- edit to your path

import os
assert os.path.isdir(os.path.join(SRC, 'graph_morphing')), SRC
print('SRC =', SRC)

fatal: destination path 'graphMorphing' already exists and is not an empty directory.
SRC = /content/graphMorphing/src


In [ ]:
# Cell 3: install dependencies (PyG + torch_cluster matched to Colab torch + gdown)
import torch
v = torch.__version__.split('+')[0]
!pip -q install torch_geometric gdown scikit-learn pandas
!pip -q install torch_cluster -f https://data.pyg.org/whl/torch-{v}+cu121.html || pip -q install torch_cluster
import torch_cluster; print('torch_cluster', torch_cluster.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# Cell 4: paths / config
import sys, json, time
from pathlib import Path
sys.path.append(SRC)

DATA_ROOT = '/content/drive/data'; os.makedirs(DATA_ROOT, exist_ok=True)  # Roman/Amazon auto-download here (gdown)
# Save results to Drive so they persist across sessions (edit as needed):
OUT_DIR = Path('/content/drive/results/hetero_distill_colab')
OUT_DIR.mkdir(parents=True, exist_ok=True)

QUICK = True   # True -> seed 0 only (smoke); False -> seeds 0..4
SEEDS_ARG = '--quick' if QUICK else '--seeds 0 1 2 3 4'
VARIANTS = 'MLP_X KD_orig_X KD_morph_X KD_morph_X_HashAdj KD_morph_X_BloomAdj'
print('QUICK =', QUICK, '| SEEDS_ARG =', SEEDS_ARG, '| OUT_DIR =', OUT_DIR)

## Cell 5: Roman-empire
Heavy: LCC label-context embeddings + morphed-graph GCN over ~22.6k nodes. Start with `QUICK=True`.

In [ ]:
!cd "{SRC}" && PYTHONPATH=. python -m graph_morphing.run_distill_hetero \
  --dataset roman_empire {SEEDS_ARG} --variants {VARIANTS} \
  --T 2 --lambda-kd 1 2 --hash-dim 256 --bloom-dim 256 --bloom-k 4 \
  --data_root "{DATA_ROOT}" --out "{OUT_DIR}/roman_empire"

## Cell 6: Amazon-ratings
Larger still. If it is too slow, keep `QUICK=True` (seed 0) or reduce `--lce_epochs`.

In [ ]:
!cd "{SRC}" && PYTHONPATH=. python -m graph_morphing.run_distill_hetero \
  --dataset amazon_ratings {SEEDS_ARG} --variants {VARIANTS} \
  --T 2 --lambda-kd 1 2 --hash-dim 256 --bloom-dim 256 --bloom-k 4 \
  --data_root "{DATA_ROOT}" --out "{OUT_DIR}/amazon_ratings"

In [ ]:
# Cell 7+8: aggregate per-seed CSVs -> combined CSVs + display table
import pandas as pd
frames = []
for ds in ['roman_empire', 'amazon_ratings']:
    p = OUT_DIR / ds / 'distill_per_seed.csv'
    if p.exists():
        frames.append(pd.read_csv(p))
if frames:
    allrows = pd.concat(frames, ignore_index=True)
    allrows.to_csv(OUT_DIR / 'hetero_distill_per_seed.csv', index=False)
    summ = (allrows.groupby(['dataset','variant','teacher_type','student_feature'])
            .agg(student_test_mean=('student_test_acc','mean'),
                 student_test_std=('student_test_acc','std'),
                 student_macro_f1=('student_macro_f1','mean'),
                 teacher_test=('teacher_test_acc','mean'))
            .reset_index())
    summ.to_csv(OUT_DIR / 'hetero_distill_summary.csv', index=False)
    order = ['MLP_X','KD_orig_X','KD_morph_X','KD_morph_X_HashAdj','KD_morph_X_BloomAdj']
    summ['variant'] = pd.Categorical(summ['variant'], order)
    display(summ.sort_values(['dataset','variant']))
    print('saved:', OUT_DIR / 'hetero_distill_summary.csv')
else:
    print('no per-seed CSVs found yet — run the dataset cells first.')

## Success conditions to check
- **S1** teacher_morph > teacher_orig
- **S2** KD_morph_X > KD_orig_X (most important)
- **S3** KD_morph_X (or +HashAdj/+BloomAdj) > MLP_X
- **S4** KD_morph_X+HashAdj / +BloomAdj > KD_morph_X (student feature morphing adds)
- **S5** BloomAdj ≳ HashAdj (secondary)

After running with `QUICK=False`, copy `OUT_DIR` into the repo's `results/hetero_distill_colab/` and
commit as a separate results commit.